In [1]:
!pip install pandas numpy opencv-python scikit-image pydicom scipy tqdm requests

In [3]:
import os
import glob
from pathlib import Path
import zipfile
import tarfile
import pandas as pd
import numpy as np
import cv2
import io
import json
import pydicom
from scipy.stats import skew, kurtosis
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import shannon_entropy
from tqdm import tqdm


In [4]:
BASE_DIR = Path.home() / "DriveTesi"

TAR_MIMIC = BASE_DIR / "workspace" / "dataset_mimic_24k.tar"
ZIP_VINBIG = BASE_DIR / "vinbigdata-chest-xray-abnormalities-detection.zip"
ZIP_PAESAGGI = BASE_DIR / "natural.zip"

CSV_MIMIC_META = BASE_DIR / "mimic-cxr-2.0.0-metadata.csv"
CSV_MIMIC_CHEX = BASE_DIR / "mimic-cxr-2.0.0-chexpert.csv"

JSON_CHECKPOINT = Path.home() / "dataset_progress.json"
PATH_OUTPUT_CSV = Path.home() / "dataset_master_15k.csv"

In [5]:
class NumpyEncoder(json.JSONEncoder):
    """Permette al JSON di salvare i numeri numpy estratti da OpenCV e Scipy"""
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

# Carica il progresso se esiste
if JSON_CHECKPOINT.exists():
    with open(JSON_CHECKPOINT, 'r') as f:
        try: progress_data = json.load(f)
        except json.JSONDecodeError: progress_data = {}
else:
    progress_data = {}

def is_image_complete(img_id, required_classes):
    """Controlla se un'immagine ha TUTTE le varianti nel JSON. Se ne manca una, torna False."""
    if img_id not in progress_data:
        return False
    stored_classes = progress_data[img_id].keys()
    for req in required_classes:
        if req not in stored_classes:
            return False
    return True

def salva_checkpoint():
    """Salva il dizionario su disco in tempo reale"""
    with open(JSON_CHECKPOINT, 'w') as f:
        json.dump(progress_data, f, cls=NumpyEncoder)

CLASSI_MIMIC = ['Perfetta', 'Negativo', 'Contrast_Stretching', 'Gray_Level_Slicing', 'Inutilizzabile']
CLASSI_VINBIG = ['Perfetta', 'Negativo', 'Contrast_Stretching', 'Gray_Level_Slicing']
CLASSI_PAESAGGI = ['Inutilizzabile']

In [6]:
def extract_features(img):
    features = {}
    pixel_vals = img.ravel()
    features['hist_mean'] = np.mean(pixel_vals)
    features['hist_std'] = np.std(pixel_vals)
    features['hist_skew'] = skew(pixel_vals)
    features['hist_kurtosis'] = kurtosis(pixel_vals)
    percentiles = np.percentile(pixel_vals, [5, 25, 50, 75, 95])
    features['perc_05'], features['perc_25'], features['perc_50'], features['perc_75'], features['perc_95'] = percentiles
    h, w = img.shape
    h_step, w_step = h // 3, w // 3
    border_means, center_mean = [], 0
    for i in range(3):
        for j in range(3):
            quadrant = img[i*h_step:(i+1)*h_step, j*w_step:(j+1)*w_step]
            q_mean = np.mean(quadrant)
            features[f'quad_{i}_{j}_mean'] = q_mean
            features[f'quad_{i}_{j}_std'] = np.std(quadrant)
            if i == 1 and j == 1: center_mean = q_mean
            else: border_means.append(q_mean)
    features['border_center_ratio'] = np.mean(border_means) / (center_mean + 1e-5)
    features['shannon_entropy'] = shannon_entropy(img)
    edges = cv2.Canny(img, 50, 150)
    features['edge_density'] = np.sum(edges > 0) / (h * w)
    glcm = graycomatrix(img, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    features['glcm_contrast'] = graycoprops(glcm, 'contrast')[0, 0]
    features['glcm_homogeneity'] = graycoprops(glcm, 'homogeneity')[0, 0]
    features['glcm_energy'] = graycoprops(glcm, 'energy')[0, 0]
    features['glcm_correlation'] = graycoprops(glcm, 'correlation')[0, 0]
    moments = cv2.moments(img)
    hu_moments = cv2.HuMoments(moments).flatten()
    for idx, hm in enumerate(hu_moments):
        features[f'hu_moment_{idx}'] = -np.sign(hm) * np.log10(np.abs(hm) + 1e-10)
    return features

In [7]:
def process_perfect_image(img, add_noise=True):
    rows = []
    rows.append({**extract_features(img), 'class_label': 'Perfetta'})
    rows.append({**extract_features(255 - img), 'class_label': 'Negativo'})
    rows.append({**extract_features(cv2.normalize(img, None, 80, 150, cv2.NORM_MINMAX)), 'class_label': 'Contrast_Stretching'})
    gamma = 0.3
    table = np.array([((i / 255.0) ** (1.0 / gamma)) * 255 for i in np.arange(0, 256)]).astype("uint8")
    rows.append({**extract_features(cv2.LUT(img, table)), 'class_label': 'Gray_Level_Slicing'})
    
    if add_noise:
        gauss = np.random.normal(0, 80, (512, 512))
        noisy_img = np.clip(img + gauss, 0, 255).astype(np.uint8)
        blurred_noisy_img = cv2.GaussianBlur(noisy_img, (15, 15), 0)
        rows.append({**extract_features(blurred_noisy_img), 'class_label': 'Inutilizzabile'})
    
    return rows

In [8]:
if TAR_MIMIC.exists():
    print("\n[1/3] Verifica MIMIC in corso...")
    df_mimic = pd.merge(pd.read_csv(str(CSV_MIMIC_META)), pd.read_csv(str(CSV_MIMIC_CHEX)), on=['subject_id', 'study_id']).set_index('dicom_id')
    with tarfile.open(str(TAR_MIMIC), 'r|') as archive:
        mimic_completate = sum(1 for v in progress_data.values() if all(c in v for c in CLASSI_MIMIC))
        
        with tqdm(total=1500, initial=mimic_completate, desc="MIMIC", unit="img", colour="green") as pbar:
            for member in archive:
                if mimic_completate >= 1500: break
                if member.isfile() and member.name.lower().endswith('.jpg'):
                    dicom_id = f"mimic_{os.path.basename(member.name).replace('.jpg', '')}"
                    
                    if is_image_complete(dicom_id, CLASSI_MIMIC):
                        continue
                        
                    if dicom_id.replace("mimic_", "") in df_mimic.index:
                        row = df_mimic.loc[dicom_id.replace("mimic_", "")]
                        if isinstance(row, pd.DataFrame): row = row.iloc[0]
                        if (row['ViewPosition'] == 'PA') and (row['No Finding'] == 1.0) and (row['Support Devices'] != 1.0):
                            file_obj = archive.extractfile(member)
                            if file_obj:
                                img = cv2.imdecode(np.frombuffer(file_obj.read(), np.uint8), cv2.IMREAD_GRAYSCALE)
                                if img is not None:
                                    img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)
                                    
                                    img_features = {r.pop('class_label'): r for r in process_perfect_image(img, add_noise=True)}
                                    progress_data[dicom_id] = img_features
                                    
                                    salva_checkpoint() 
                                    mimic_completate += 1
                                    pbar.update(1)


[1/3] Verifica MIMIC in corso...


MIMIC: 100%|██████████| 1500/1500 [00:00<?, ?img/s]


In [9]:
if ZIP_VINBIG.exists():
    print("\n[2/3] Verifica VinBigData in corso (Lettura Sequenziale Rapida)...")
    with zipfile.ZipFile(str(ZIP_VINBIG), 'r') as archive:
        file_nel_zip = set(archive.namelist())
        if 'train.csv' in file_nel_zip:
            with archive.open('train.csv') as f:
                df_vin = pd.read_csv(f)
            
            vin_perfect_ids = df_vin[df_vin['class_id'] == 14]['image_id'].unique()[:1500]
            target_paths = {f"train/{img_id}.dicom" for img_id in vin_perfect_ids}
            
            vin_completate = sum(1 for img_id in vin_perfect_ids if is_image_complete(f"vin_{img_id}", CLASSI_VINBIG))
            
            with tqdm(total=1500, initial=vin_completate, desc="VinBigData", unit="img", colour="blue") as pbar:
                for info in archive.infolist():
                    if vin_completate >= 1500: 
                        break
                        
                    if info.filename in target_paths:
                        img_id = info.filename.replace("train/", "").replace(".dicom", "")
                        dict_id = f"vin_{img_id}"
                        
                        if is_image_complete(dict_id, CLASSI_VINBIG):
                            continue
                            
                        pbar.set_postfix_str(f"Scan: {info.filename[-15:]}")
                        try:
                            with archive.open(info) as file_obj:
                                img_data = file_obj.read()
                                
                            dicom_file = pydicom.dcmread(io.BytesIO(img_data))
                            img = dicom_file.pixel_array.astype(float)
                            img = ((img - np.min(img)) / (np.max(img) - np.min(img) + 1e-5) * 255).astype(np.uint8)
                            img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)
                            
                            img_features = {r.pop('class_label'): r for r in process_perfect_image(img, add_noise=False)}
                            progress_data[dict_id] = img_features
                            
                            salva_checkpoint()
                            vin_completate += 1
                            pbar.update(1)
                        except Exception: 
                            pass


[2/3] Verifica VinBigData in corso (Lettura Sequenziale Rapida)...


VinBigData: 100%|██████████| 1500/1500 [2:25:03<00:00,  6.50s/img, Scan: e329307a4.dicom]  


In [10]:
if ZIP_PAESAGGI.exists():
    print("\n[3/3] Verifica Paesaggi in corso (Lettura Sequenziale)...")
    with zipfile.ZipFile(str(ZIP_PAESAGGI), 'r') as archive:
        
        pae_completate = sum(1 for k, v in progress_data.items() if k.startswith("pae_") and is_image_complete(k, CLASSI_PAESAGGI))
        
        with tqdm(total=1500, initial=pae_completate, desc="Paesaggi", unit="img", colour="yellow") as pbar:
            for info in archive.infolist():
                if pae_completate >= 1500:
                    break
                
                if info.filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    dict_id = f"pae_{os.path.basename(info.filename)}"
                    
                    if is_image_complete(dict_id, CLASSI_PAESAGGI):
                        continue
                        
                    pbar.set_postfix_str(f"Scan: {info.filename[-20:]}")
                    try:
                        with archive.open(info) as file_obj:
                            img_data = file_obj.read()
                            
                        img = cv2.imdecode(np.frombuffer(img_data, np.uint8), cv2.IMREAD_GRAYSCALE)
                        if img is not None:
                            img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)
                            
                            feat = extract_features(img)
                            progress_data[dict_id] = {'Inutilizzabile': feat}
                            
                            salva_checkpoint()
                            pae_completate += 1
                            pbar.update(1)
                    except Exception: 
                        pass


[3/3] Verifica Paesaggi in corso (Lettura Sequenziale)...


Paesaggi: 100%|██████████| 1500/1500 [12:05<00:00,  2.07img/s, Scan: ges/car/car_0772.jpg]


In [11]:
print("\nGenerazione del file CSV finale dal JSON in corso...")
final_rows = []
for img_id, classi in progress_data.items():
    for label, features in classi.items():
        row = features.copy()
        row['class_label'] = label
        final_rows.append(row)

final_df = pd.DataFrame(final_rows)
final_df.to_csv(str(PATH_OUTPUT_CSV), index=False)
print(f"\nDataset salvato in: {PATH_OUTPUT_CSV}")
print("\nDistribuzione finale:")
print(final_df['class_label'].value_counts())


Generazione del file CSV finale dal JSON in corso...

Dataset salvato in: /home/andy/dataset_master_15k.csv

Distribuzione finale:
class_label
Perfetta               3000
Negativo               3000
Contrast_Stretching    3000
Gray_Level_Slicing     3000
Inutilizzabile         3000
Name: count, dtype: int64
